# Extracción de características: BoW & TFIDF

## BoW: bolsa de palabras

 Técnica que transforma texto no estructurado en vectores numéricos.

 Los modelos de BoW consideran todas las palabras presentes en un texto, jerarquizándolas en función de su frecuencia e ignorando su orden y el contexto en el que aparecen.

In [ ]:
# Importar librerías
import numpy as np
import sklearn
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files


In [ ]:
# Cargamos el archivo con los datos

df = pd.read_csv('20000-en.csv', encoding='utf-8') #, on_bad_lines='skip')
df.tail(10)

In [ ]:
#Usamos los textos que están en la columna Text y los guardamos en un DF

datos = df.Text.to_list()
datos

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Instanciar el módulo para vectorizar los textos
vectorizador =  CountVectorizer()


# Vectorizar las palabras a partir de sus frecuencias
bow = vectorizador.fit_transform(datos)

In [ ]:
print(bow)

In [ ]:
# Vocabulario

print(vectorizador.vocabulary_)   #notar que el ID no corresponde a la frecuencia, sino al índice en la matriz

In [ ]:
# Vocabulario (ordenar por índices)

print (sorted(vectorizador.vocabulary_.items(), key=lambda x: x[1])) # reverse=True para orden inverso

In [ ]:
# Obtener el índice de una palabra N

print(vectorizador.vocabulary_['ray'])

## Matriz por frecuencias

Si queremos ver las frecuencias de cada palabra, debemos sumar las frecuencias individuales y asociar el resultado a un índice. Posteriormente, podemos guardar esa información en una variable o en un DF.

In [ ]:
#Creamos un DF para almacenar las frecuencias

freqs = bow.sum(axis=0)
words = vectorizador.get_feature_names_out()
df_freqs = pd.DataFrame((freqs.T), index=words, columns=['freq'])
df_freqs.sort_values(by=['freq'], ascending=False)


## Visualización de las N palabras más frecuentes

In [ ]:
df_freqs.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))

## Limpieza del corpus  

* Eliminación de stopwords

* Matriz más pequeña

In [ ]:
# Usamos una lista de stopwords para pasarla por el vectorizador

vectorizador_clean =  CountVectorizer(stop_words=list(pd.read_csv('stopw-short-en.txt', header=None)[0].tolist()), analyzer = 'word')
bow_clean = vectorizador_clean.fit_transform(datos)
print(vectorizador_clean.vocabulary_)


In [ ]:
# Primera comparación -> reducción de vocabulario

print(vectorizador.vocabulary_)
print(vectorizador_clean.vocabulary_)

In [ ]:
# Segunda comparación -> modificación de índices por reducción

print(vectorizador.vocabulary_['ray'])
print(vectorizador_clean.vocabulary_['ray'])

In [ ]:
# Creación de DF para frecuencias

freqs_clean = bow_clean.sum(axis=0)
words_clean = vectorizador_clean.get_feature_names_out()
df_freqs_clean = pd.DataFrame((freqs_clean.T), index=words_clean, columns=['freq'])
df_freqs_clean.sort_values(by=['freq'], ascending=False)

In [ ]:
# Visualización de N palabras más frecuentes con corpus limpio

df_freqs_clean.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))

## Ejemplo más realista

Cuando se trabaja con datos reales, es casi un hecho que la cantidad de datos sea de millones, tanto de palabras como de documentos. Ello, además de ser inviable para un humano o, incluso, para algunos softwares, representa un problema muy importante: las matrices que, de por sí son N-dimensionales, se vuelven gigantescas y, en la mayoría de los casos, con muchos ceros y pocos unos. Para minimizar este tipo de problemáticas, se trabaja con otros métodos, los cuales se verán más adelante.

En todo caso, veremos un ejemplo que toma como corpus los datos de la evaluación, es decir:


corpus size = $\sum_{w1}^{wN}$ $\forall$ $S_i$







In [ ]:
# Cargamos el archivo con los datos

df = pd.read_csv('sentiment.csv')
print(df.head(5))

In [ ]:
#Usamos los textos que están en la columna Text y los guardamos en un DF

corpus = df.Text.tolist()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Instanciar el módulo para vectorizar los textos
vectorizador =  CountVectorizer()


# Vectorizar las palabras a partir de sus frecuencias
bow = vectorizador.fit_transform(corpus)

In [ ]:
# Vector
print(bow)

In [ ]:
# Vocabulario

print(vectorizador.vocabulary_)


In [ ]:
#Creamos un DF para almacenar las frecuencias

freqs = bow.sum(axis=0)
words = vectorizador.get_feature_names_out()
df_freqs = pd.DataFrame((freqs.T), index=words, columns=['freq'])
df_freqs.sort_values(by=['freq'], ascending=False)

In [ ]:
df_freqs.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))
plt.savefig('freqs.png')
files.download('freqs.png')

## Limpieza del corpus

In [ ]:
vectorizador_clean =  CountVectorizer(stop_words=list(pd.read_csv('stopw-large-en.txt', header=None)[0].tolist()), analyzer = 'word')
bow_clean = vectorizador_clean.fit_transform(corpus)
print(vectorizador_clean.vocabulary_)

In [ ]:
# Primera comparación -> reducción de vocabulario

print(vectorizador.vocabulary_)
print(vectorizador_clean.vocabulary_)

In [ ]:
# Segunda comparación -> modificación de índices por reducción

print(vectorizador.vocabulary_['irony'])
print(vectorizador_clean.vocabulary_['irony'])

In [ ]:
# Creación de DF para frecuencias

freqs_clean = bow_clean.sum(axis=0)
words_clean = vectorizador_clean.get_feature_names_out()
df_freqs_clean = pd.DataFrame((freqs_clean.T), index=words_clean, columns=['freq'])
df_freqs_clean.sort_values(by=['freq'], ascending=False)

In [ ]:
# Visualización de N palabras más frecuentes con corpus limpio

df_freqs_clean.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))
plt.savefig('freqs_clean.png')
files.download('freqs_clean.png')

## Vectorización más fina

* Considerar representación de ngramas, donde N = 1 ... n

In [ ]:
# Se añade un parámetro para definir el rango de N

vectorizador_clean_n =  CountVectorizer(stop_words=list(pd.read_csv('stopw-large-en.txt', header=None)[0].tolist()), analyzer = 'word',
                                      ngram_range=(1,3))
bow_clean_n = vectorizador_clean_n.fit_transform(corpus)


In [ ]:
# Guardamos los resultados en un nuevo DF

freqs_clean_n = bow_clean_n.sum(axis=0)
words_clean_n = vectorizador_clean_n.get_feature_names_out()
df_freqs_clean_n = pd.DataFrame((freqs_clean_n.T), index=words_clean_n, columns=['freq'])
df_freqs_clean_n.sort_values(by=['freq'], ascending=False)

In [ ]:
# Visualizamos

df_freqs_clean_n.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))
plt.savefig('freqs_clean_n.png')
files.download('freqs_clean_n.png')

Existen otros parámetros que podemos usar para que la dimensionalidad de la matriz no sea tan grande. Asimismo, se puede definir cuál es el máximo de palabras que se deben extraer, etc. Todo ello está en la documentación de Vectorizer en [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

Por ejemplo, con **max_features = INT**, se establece cuántas palabras se considerarán.

In [ ]:
vectorizador_clean_n =  CountVectorizer(stop_words=list(pd.read_csv('stopw-large-en.txt', header=None)[0].tolist()), analyzer = 'word',
                                      ngram_range=(1,3), max_features = 50)
bow_clean_n = vectorizador_clean_n.fit_transform(corpus)

In [ ]:
freqs_clean_n = bow_clean_n.sum(axis=0)
words_clean_n = vectorizador_clean_n.get_feature_names_out()
df_freqs_clean_n = pd.DataFrame((freqs_clean_n.T), index=words_clean_n, columns=['freq'])
df_freqs_clean_n.sort_values(by=['freq'], ascending=False)

Si bien BoW es un método eficiente para representar características explícitas en un corpus, impide observar comportamientos que están dados por el contexto o, incluso, por la baja frecuencia. Piensen, por ejemplo, en un hilo de comentarios en donde la palabra 'indio' solo aparece una vez, pero su aparición constituye el detonante para generar discurso de odio. En estos casos, es mejor afinar la aproximación con técnicas más complejas. Una de ellas es **TFIDF** (term frequency - inverse document frequency).

# TF-IDF

* Mide qué tan común o raro es un término $t$ en toda la colección de documentos $D$:

* $TF(t,d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$


donde $f_{t,d}$ es el número de veces que aparece el término $t$ en el documento $d$, entre la suma de las frecuencias de todos los términos dentro del mismo documento $d$.

In [ ]:
# Para hacerlo, hay que importar el método TFIDF de sklearn

from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
# Trabajamos sobre los datos de la variable corpus

#Instanciamos el método
idf_vectorizer = TfidfVectorizer()

# Vectorizamos
corpus_idf = idf_vectorizer.fit_transform(corpus)


In [ ]:
print(idf_vectorizer.vocabulary_)


In [ ]:
freqs_idf = corpus_idf.sum(axis=0)
words_idf = idf_vectorizer.get_feature_names_out()
df_freqs_idf = pd.DataFrame((freqs_idf.T), index=words_idf, columns=['freq'])
df_freqs_idf.sort_values(by=['freq'], ascending=False)

In [ ]:
df_freqs_idf.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))
plt.savefig('freqs_idf.png')
files.download('freqs_idf.png')

In [ ]:
#Instanciamos el método
idf_clean = TfidfVectorizer(stop_words=list(pd.read_csv('stopw-large-en.txt', header=None)[0].tolist()), analyzer = 'word')

# Vectorizamos
corpus_idf_clean = idf_clean.fit_transform(corpus)


In [ ]:
freqs_idf_clean = corpus_idf_clean.sum(axis=0)
words_idf_clean = idf_clean.get_feature_names_out()
df_freqs_idf_clean = pd.DataFrame((freqs_idf_clean.T), index=words_idf_clean, columns=['freq'])
df_freqs_idf_clean.sort_values(by=['freq'], ascending=False)

In [ ]:
df_freqs_idf_clean.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8))
plt.savefig('freqs_idf_clean.png')
files.download('freqs_idf_clean.png')